# 7-7_2 Transformer の学習・推論を今の書き方で行う

`7-7_transformer_training_inference.ipynb` は、IMDb の DataLoader と Transformer で感情分析を学習し、テストデータの正解率を出し、`<cls>` の Attention を色で見せる。このファイルは、その流れのまま、ミニバッチの作り方と学習ループを今の `Dataset` と `DataLoader` で書く。

モデルの計算は 7-7 と同じ `utils/transformer.py` の `TransformerClassification` である。単語 ID、`<cls>`、`<eos>`、長さ 256 のパディングは `7-5_2_IMDb_Dataset_DataLoader.ipynb` と同じである。7-7 のバッチサイズは 64 なので、ここでも 64 にする。


# 7-7_2 学習目標

1.	7-7 と同じ Transformer を、今の DataLoader が返すミニバッチで学習する
2.	テストデータの正解率と、`<cls>` から見た Attention の可視化を同じ手順で出す


# 事前準備

7-5 で作った次のファイルを使う。

- `data/IMDb_train.tsv`
- `data/IMDb_test.tsv`
- `data/wiki-news-300d-1M.vec`

モデルのクラスは `utils/transformer.py` に入っている。fastText の読み込みには gensim 3.8.3 を使う。パッケージは Docker の JupyterLab（http://127.0.0.1:8887/lab）に入っている。


In [1]:
# パッケージのimport
import random
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from gensim.models import KeyedVectors
from torch.utils.data import DataLoader, Dataset


In [2]:
# 乱数のシードを設定
torch.manual_seed(1234)
np.random.seed(1234)
random.seed(1234)


# DatasetとDataLoaderを作成

7-7 は次の関数で DataLoader を受け取っていた。

```python
from utils.dataloader import get_IMDb_DataLoaders_and_TEXT

train_dl, val_dl, test_dl, TEXT = get_IMDb_DataLoaders_and_TEXT(
    max_length=256, batch_size=64)

dataloaders_dict = {"train": train_dl, "val": val_dl}
```

この関数の中身は `torchtext.data.Field` と `Iterator` である。学習ループは `batch.Text[0]` を文章の ID、`batch.Label` をラベル、`TEXT.vocab.vectors` を単語ベクトルとして使っている。

ここでは 7-5_2 と同じ手順で、その3つを作る。tsv の前処理、80:20 の分割、語彙の並び、`<cls>` と `<eos>` を含む長さ 256 は 7-5 に合わせる。訓練データは各エポックでシャッフルし、検証データとテストデータはデータセットの順のまま取り出す。


In [3]:
import string
import re


def preprocessing_text(text):
    """7-5 の preprocessing_text と同じ。"""
    text = re.sub("<br />", "", text)
    for p in string.punctuation:
        if (p == ".") or (p == ","):
            continue
        text = text.replace(p, " ")
    text = text.replace(".", " . ")
    text = text.replace(",", " , ")
    return text


def tokenizer_with_preprocessing(text):
    """7-5 の tokenizer_with_preprocessing と同じ。"""
    text = preprocessing_text(text)
    return text.strip().split()


print(tokenizer_with_preprocessing("I like cats."))


['I', 'like', 'cats', '.']


In [4]:
class TextDataset(Dataset):
    """tsv の1行を (単語リスト, ラベル) として持つ。7-7 が使う TabularDataset に相当する。"""

    def __init__(self, samples):
        self.samples = list(samples)

    @classmethod
    def from_tsv(cls, path, tokenizer):
        samples = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                text, label, *_rest = line.rstrip("\n").split("\t")
                tokens = tokenizer(text)
                # Field は tokenize のあとで lower=True を各単語にかける
                tokens = [token.lower() for token in tokens]
                samples.append((tokens, label))
        return cls(samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


train_val_ds = TextDataset.from_tsv("./data/IMDb_train.tsv", tokenizer_with_preprocessing)
test_ds = TextDataset.from_tsv("./data/IMDb_test.tsv", tokenizer_with_preprocessing)

print("訓練および検証のデータ数", len(train_val_ds))
print("テストのデータ数", len(test_ds))


訓練および検証のデータ数 25000
テストのデータ数 25000


In [5]:
def split_train_val(dataset, split_ratio=0.8, seed=1234):
    """7-7 の Dataset.split(split_ratio=0.8, random_state=random.seed(1234)) に相当する。"""
    random.seed(seed)
    order = random.sample(range(len(dataset)), len(dataset))
    train_len = int(round(split_ratio * len(dataset)))
    train_samples = [dataset[index] for index in order[:train_len]]
    val_samples = [dataset[index] for index in order[train_len:]]
    return TextDataset(train_samples), TextDataset(val_samples)


train_ds, val_ds = split_train_val(train_val_ds)

print("訓練データの数", len(train_ds))
print("検証データの数", len(val_ds))


訓練データの数 20000
検証データの数 5000


In [6]:
english_fasttext = KeyedVectors.load_word2vec_format(
    "data/wiki-news-300d-1M.vec", binary=False)


def build_vocab(dataset, min_freq=10):
    """7-7 の TEXT.build_vocab(train_ds, min_freq=10) に相当する。"""
    freqs = Counter()
    for tokens, _label in dataset:
        freqs.update(tokens)

    specials = ["<unk>", "<pad>", "<cls>", "<eos>"]
    for token in specials:
        if token in freqs:
            del freqs[token]

    words = [word for word, freq in freqs.items() if freq >= min_freq]
    words.sort(key=lambda word: (-freqs[word], word))
    itos = specials + words
    stoi = {word: index for index, word in enumerate(itos)}
    return stoi, itos


def build_vector_table(keyed_vectors, stoi):
    """7-7 の TEXT.vocab.vectors に相当する行列を作る。"""
    table = torch.zeros(len(stoi), keyed_vectors.vector_size)
    for word, index in stoi.items():
        if word in keyed_vectors:
            table[index] = torch.tensor(keyed_vectors[word].copy(), dtype=table.dtype)
    return table


stoi, itos = build_vocab(train_ds, min_freq=10)
vectors = build_vector_table(english_fasttext, stoi)

print("語彙ベクトルのサイズ:", vectors.shape)
print("<pad> の ID:", stoi["<pad>"])


語彙ベクトルのサイズ: torch.Size([17892, 300])
<pad> の ID: 1


In [7]:
max_length = 256
content_length = max_length - 2  # <cls> と <eos> の分を除いた本文の長さ
pad_id = stoi["<pad>"]
unk_id = stoi["<unk>"]
cls_id = stoi["<cls>"]
eos_id = stoi["<eos>"]


def collate_batch(batch):
    """7-7 の batch.Text[0] と batch.Label に相当するテンソルを返す。"""
    text_batch = []
    lengths = []
    labels = []

    for tokens, label in batch:
        ids = [stoi.get(token, unk_id) for token in tokens[:content_length]]
        ids = [cls_id] + ids + [eos_id]
        length = len(ids)
        ids += [pad_id] * (max_length - len(ids))

        text_batch.append(torch.tensor(ids, dtype=torch.long))
        lengths.append(length)
        labels.append(int(label))

    text = torch.stack(text_batch)
    lengths = torch.tensor(lengths, dtype=torch.long)
    labels = torch.tensor(labels, dtype=torch.long)
    return text, lengths, labels


# 7-7 は batch_size=64。train=True はシャッフルする、という意味だった
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_batch)
val_dl = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_batch)
test_dl = DataLoader(test_ds, batch_size=64, shuffle=False, collate_fn=collate_batch)

dataloaders_dict = {"train": train_dl, "val": val_dl}

text, lengths, labels = next(iter(val_dl))
print("文章の ID:", text.shape)
print("ラベル:", labels.shape)


文章の ID: torch.Size([64, 256])
ラベル: torch.Size([64])


# ネットワークモデルの作成

7-7 は `TEXT.vocab.vectors` を `TransformerClassification` に渡している。ここでは直前に作った `vectors` を渡す。`TransformerBlock` の全結合層だけを Kaiming の正規分布で初期化し直す処理も、7-7 と同じである。


In [8]:
from utils.transformer import TransformerClassification

net = TransformerClassification(
    text_embedding_vectors=vectors, d_model=300, max_seq_len=256, output_dim=2)


def weights_init(m):
    classname = m.__class__.__name__
    if classname.find("Linear") != -1:
        nn.init.kaiming_normal_(m.weight)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0.0)


net.train()
net.net3_1.apply(weights_init)
net.net3_2.apply(weights_init)

print("ネットワーク設定完了")


ネットワーク設定完了


# 損失関数と最適化手法を定義


In [9]:
# 損失関数の設定
criterion = nn.CrossEntropyLoss()

# 最適化手法の設定
learning_rate = 2e-5
optimizer = optim.Adam(net.parameters(), lr=learning_rate)


# 学習・検証を実施

7-7 の学習ループは、各ミニバッチで `batch.Text[0]` と `batch.Label` を取り、`<pad>` 以外を True にしたマスクを Transformer に渡している。ここでは DataLoader が返す `text` と `labels` を、その2つの位置に置く。

エポック数は 10 である。CPU では 7-7 に書かれている時間より長くかかる。


In [10]:
def train_model(net, dataloaders_dict, criterion, optimizer, num_epochs):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print("使用デバイス：", device)
    print("-----start-------")
    net.to(device)

    torch.backends.cudnn.benchmark = True

    for epoch in range(num_epochs):
        for phase in ["train", "val"]:
            if phase == "train":
                net.train()
            else:
                net.eval()

            epoch_loss = 0.0
            epoch_corrects = 0

            for text, _lengths, labels_batch in dataloaders_dict[phase]:
                # 7-7 の batch.Text[0] と batch.Label
                inputs = text.to(device)
                labels = labels_batch.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == "train"):
                    # 7-7 は (inputs != 1)。<pad> の ID は 1
                    input_mask = (inputs != pad_id)

                    outputs, _, _ = net(inputs, input_mask)
                    loss = criterion(outputs, labels)

                    _, preds = torch.max(outputs, 1)

                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                    epoch_loss += loss.item() * inputs.size(0)
                    epoch_corrects += torch.sum(preds == labels.data)

            epoch_loss = epoch_loss / len(dataloaders_dict[phase].dataset)
            epoch_acc = epoch_corrects.double() / len(dataloaders_dict[phase].dataset)

            print("Epoch {}/{} | {:^5} |  Loss: {:.4f} Acc: {:.4f}".format(
                epoch + 1, num_epochs, phase, epoch_loss, epoch_acc))

    return net


In [11]:
# 学習・検証を実行する
num_epochs = 10
net_trained = train_model(
    net, dataloaders_dict, criterion, optimizer, num_epochs=num_epochs)


使用デバイス： cpu
-----start-------
Epoch 1/10 | train |  Loss: 0.6179 Acc: 0.6445
Epoch 1/10 |  val  |  Loss: 0.4527 Acc: 0.7928
Epoch 2/10 | train |  Loss: 0.4452 Acc: 0.7952
Epoch 2/10 |  val  |  Loss: 0.4291 Acc: 0.8088
Epoch 3/10 | train |  Loss: 0.4132 Acc: 0.8131
Epoch 3/10 |  val  |  Loss: 0.4118 Acc: 0.8148
Epoch 4/10 | train |  Loss: 0.3921 Acc: 0.8254
Epoch 4/10 |  val  |  Loss: 0.3880 Acc: 0.8236
Epoch 5/10 | train |  Loss: 0.3786 Acc: 0.8340
Epoch 5/10 |  val  |  Loss: 0.3764 Acc: 0.8284
Epoch 6/10 | train |  Loss: 0.3664 Acc: 0.8385
Epoch 6/10 |  val  |  Loss: 0.3715 Acc: 0.8366
Epoch 7/10 | train |  Loss: 0.3584 Acc: 0.8426
Epoch 7/10 |  val  |  Loss: 0.3804 Acc: 0.8288
Epoch 8/10 | train |  Loss: 0.3515 Acc: 0.8472
Epoch 8/10 |  val  |  Loss: 0.3553 Acc: 0.8428
Epoch 9/10 | train |  Loss: 0.3414 Acc: 0.8525
Epoch 9/10 |  val  |  Loss: 0.3485 Acc: 0.8464
Epoch 10/10 | train |  Loss: 0.3355 Acc: 0.8548
Epoch 10/10 |  val  |  Loss: 0.3667 Acc: 0.8430


# テストデータでの正解率を求める


In [12]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

net_trained.eval()
net_trained.to(device)

epoch_corrects = 0

for text, _lengths, labels_batch in test_dl:
    inputs = text.to(device)
    labels = labels_batch.to(device)

    with torch.set_grad_enabled(False):
        input_mask = (inputs != pad_id)
        outputs, _, _ = net_trained(inputs, input_mask)
        _, preds = torch.max(outputs, 1)
        epoch_corrects += torch.sum(preds == labels.data)

epoch_acc = epoch_corrects.double() / len(test_dl.dataset)

print("テストデータ{}個での正解率：{:.4f}".format(len(test_dl.dataset), epoch_acc))


テストデータ25000個での正解率：0.8444


# Attentionの可視化で判定根拠を探る

7-7 は、各 TransformerBlock の Attention のうち、文頭の `<cls>` が他の単語を見た重みを取り出す。値が大きい単語ほど背景を濃い赤にする。単語の文字列は `TEXT.vocab.itos` から引いていた。ここでは `itos` を使う。


In [13]:
def highlight(word, attn):
    """Attentionの値が大きいと文字の背景が濃い赤になるhtmlを出力させる関数"""
    html_color = "#%02X%02X%02X" % (
        255, int(255 * (1 - attn)), int(255 * (1 - attn)))
    return '<span style="background-color: {}"> {}</span>'.format(html_color, word)


def mk_html(index, text, labels, preds, normlized_weights_1, normlized_weights_2, itos):
    """HTMLデータを作成する。7-7 の batch.Text[0]、batch.Label、TEXT.vocab.itos を引数で受け取る。"""
    sentence = text[index]
    label = labels[index]
    pred = preds[index]

    # 0番目の <cls> の Attention を、最大値で 0 から 1 にする
    attens1 = normlized_weights_1[index, 0, :]
    attens1 = attens1 / attens1.max()

    attens2 = normlized_weights_2[index, 0, :]
    attens2 = attens2 / attens2.max()

    if int(label) == 0:
        label_str = "Negative"
    else:
        label_str = "Positive"

    if int(pred) == 0:
        pred_str = "Negative"
    else:
        pred_str = "Positive"

    html = "正解ラベル：{}<br>推論ラベル：{}<br><br>".format(label_str, pred_str)

    html += "[TransformerBlockの1段目のAttentionを可視化]<br>"
    for word, attn in zip(sentence, attens1):
        html += highlight(itos[int(word)], attn)
    html += "<br><br>"

    html += "[TransformerBlockの2段目のAttentionを可視化]<br>"
    for word, attn in zip(sentence, attens2):
        html += highlight(itos[int(word)], attn)
    html += "<br><br>"

    return html


In [14]:
from IPython.display import HTML

batch_text, _batch_lengths, batch_labels = next(iter(test_dl))

inputs = batch_text.to(device)
input_mask = (inputs != pad_id)

outputs, normlized_weights_1, normlized_weights_2 = net_trained(inputs, input_mask)
_, preds = torch.max(outputs, 1)

index = 3
html_output = mk_html(
    index, batch_text, batch_labels, preds,
    normlized_weights_1, normlized_weights_2, itos)
HTML(html_output)


In [15]:
index = 61
html_output = mk_html(
    index, batch_text, batch_labels, preds,
    normlized_weights_1, normlized_weights_2, itos)
HTML(html_output)


以上
